In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd

try:
    import blpapi
except ModuleNotFoundError as error:
    raise RuntimeError(
        "Bloomberg BLPAPI is required. Install it from Bloomberg's package index "
        "and run this notebook with Bloomberg Terminal/API access."
    ) from error

SECURITY = "AAPL US Equity"
FIELDS = ["PX_OPEN", "PX_HIGH", "PX_LOW", "PX_LAST", "VOLUME"]
DATA_DIR = Path("data")


def _element_value(row, field):
    if not row.hasElement(field) or row.getElement(field).isNull():
        return None
    return row.getElementAsFloat(field)


def fetch_historical_prices(security=SECURITY, start_date=None, end_date=None):
    """Pull daily Bloomberg history and return it as a DataFrame."""
    today = date.today()
    start_date = start_date or date(today.year, 1, 1)
    end_date = end_date or today

    session = blpapi.Session()
    if not session.start():
        raise RuntimeError("Could not start Bloomberg session. Is Bloomberg Terminal running?")

    try:
        if not session.openService("//blp/refdata"):
            raise RuntimeError("Could not open Bloomberg reference-data service.")

        request = session.getService("//blp/refdata").createRequest("HistoricalDataRequest")
        request.getElement("securities").appendValue(security)
        for field in FIELDS:
            request.getElement("fields").appendValue(field)
        request.set("startDate", start_date.strftime("%Y%m%d"))
        request.set("endDate", end_date.strftime("%Y%m%d"))
        request.set("periodicitySelection", "DAILY")
        session.sendRequest(request)

        records = []
        while True:
            event = session.nextEvent()
            for message in event:
                if not message.hasElement("securityData"):
                    continue
                field_data = message.getElement("securityData").getElement("fieldData")
                for index in range(field_data.numValues()):
                    row = field_data.getValueAsElement(index)
                    record = {"date": row.getElementAsDatetime("date")}
                    record.update({field.lower(): _element_value(row, field) for field in FIELDS})
                    records.append(record)
            if event.eventType() == blpapi.Event.RESPONSE:
                break

        return pd.DataFrame(records).set_index("date").sort_index()
    finally:
        session.stop()


prices = fetch_historical_prices()
DATA_DIR.mkdir(parents=True, exist_ok=True)
output_path = DATA_DIR / "aapl_us_equity_2026.csv"
prices.to_csv(output_path)
print(f"Saved {len(prices):,} rows to {output_path}")
prices.tail()


ModuleNotFoundError: No module named 'blpapi'